# ── 0. 配置区 ──────────────────────────────────────────────

In [1]:
from pathlib import Path

# FiftyOne 数据集 + 预测字段
DATASET_NAME = "sahi_null_run_whole_rawData_v5_ms2_0605_0923"
PRED_FIELD   = "pred_yolo11n_20pct_null_images_add_rawData_batch_8_final"
VERSION      = "sahi_null_run_whole_rawData_v5"

# 文件名年份
YEAR = 2024

# 置信度统计阈值
HIGH_CONF_THR = 0.85
LOW_CONF_THR  = 0.50

# 输出路径
OUT_DIR = Path("./_deploy_exports") / VERSION / DATASET_NAME / PRED_FIELD
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_TABLE1_IMG  = OUT_DIR / "table_1_per_image_inference.csv"
OUT_TABLE1B_DET = OUT_DIR / "table_1b_per_detection_long.csv"
OUT_TABLE2_TXF  = OUT_DIR / "table_2_per_timepoint_focus.csv"
OUT_TABLE3_FUSE = OUT_DIR / "table_3_per_timepoint_fused_max.csv"

# ── 1. 初始化日志 ──────────────────────────────────────────

In [2]:
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")

2026-03-16 14:52:55,235 [INFO] ============ Notebook logging 初始化完成 ============


# ── 2. Step 1：导出预测结果到 CSV（表1 + 表1b）──────────────

In [3]:
# ── Step 1：从 FiftyOne dataset 导出 per-image + per-detection CSV ─
# 输入：DATASET_NAME, PRED_FIELD
# 输出：OUT_TABLE1_IMG, OUT_TABLE1B_DET

logger.info("Step 1 开始：导出预测结果 CSV")

import fiftyone as fo
from functools import partial
from ty_run_tools import export_per_image_and_detection_dfs, parse_dt_focus

parse_dt_fn = partial(parse_dt_focus, year=YEAR)

try:
    ds = fo.load_dataset(DATASET_NAME)
    logger.info(f"数据集加载成功：{DATASET_NAME}，共 {len(ds)} 张")
except Exception as e:
    logger.error(f"数据集加载失败: {e}")
    raise

if PRED_FIELD not in ds.get_field_schema():
    raise ValueError(f"pred_field '{PRED_FIELD}' 不存在，可用字段: {list(ds.get_field_schema())}")

try:
    img_df, det_df = export_per_image_and_detection_dfs(
        ds, PRED_FIELD, parse_dt_fn, HIGH_CONF_THR, LOW_CONF_THR
    )
    img_df.to_csv(OUT_TABLE1_IMG,  index=False)
    det_df.to_csv(OUT_TABLE1B_DET, index=False)
    logger.info(f"Step 1 完成：table1={img_df.shape}, table1b={det_df.shape}")
    logger.info(f"已保存: {OUT_TABLE1_IMG}, {OUT_TABLE1B_DET}")
except Exception as e:
    logger.error(f"导出失败: {e}")
    raise

2026-03-16 14:52:55,242 [INFO] Step 1 开始：导出预测结果 CSV
/home/tianqi/miniconda3/envs/fif/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-16 14:52:56,085 [ERROR] 数据集加载失败: Dataset sahi_null_run_whole_rawData_v5_ms2_0605_0923 not found


DatasetNotFoundError: Dataset sahi_null_run_whole_rawData_v5_ms2_0605_0923 not found

# ── 3. Step 2：按 timepoint × focus 聚合（表2）──────────────

In [ ]:
# ── Step 2：从表1 CSV 聚合 per-timepoint × focus 统计 ─────────
# 输入：OUT_TABLE1_IMG
# 输出：OUT_TABLE2_TXF

logger.info("Step 2 开始：聚合 per-timepoint × focus")

import pandas as pd
import numpy as np

df = pd.read_csv(OUT_TABLE1_IMG)
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["has_detection"] = (df["pred_count"].fillna(0) > 0).astype(int)

gcols = ["datetime", "focus"]
agg = df.groupby(gcols, as_index=False).agg(
    capture_date=("date", "first"),
    capture_time=("time", "first"),
    frame_count=("filepath", "count"),
    total_pred_count=("pred_count", "sum"),
    mean_pred_count=("pred_count", "mean"),
    median_pred_count=("pred_count", "median"),
    std_pred_count=("pred_count", "std"),
    confidence_mass_sum=("conf_mass", "sum"),
    confidence_mass_mean=("conf_mass", "mean"),
    confidence_mass_std=("conf_mass", "std"),
    frames_with_detection=("has_detection", "sum"),
)
agg["detection_rate"] = agg["frames_with_detection"] / agg["frame_count"]
extra = df.groupby(gcols, as_index=False).agg(
    conf_q50_time_mean=("conf_q50", "mean"),
    conf_iqr_time_mean=("conf_iqr", "mean"),
    conf_std_time_mean=("conf_std", "mean"),
)
agg = agg.merge(extra, on=gcols, how="left")
for c in ["std_pred_count", "confidence_mass_std"]:
    agg[c] = agg[c].fillna(0.0)

try:
    agg.to_csv(OUT_TABLE2_TXF, index=False)
    logger.info(f"Step 2 完成：table2={agg.shape}，保存 {OUT_TABLE2_TXF}")
except Exception as e:
    logger.error(f"保存 table2 失败: {e}")

# ── 4. Step 3：按 timepoint 融合（取 max confidence_mass）（表3）─

In [ ]:
# ── Step 3：从表2 CSV 按 timepoint 融合（取 confidence_mass_sum 最大的 focus）─
# 输入：OUT_TABLE2_TXF
# 输出：OUT_TABLE3_FUSE

logger.info("Step 3 开始：按 timepoint 融合（max confidence_mass）")

df2 = pd.read_csv(OUT_TABLE2_TXF)
df2["datetime"] = pd.to_datetime(df2["datetime"], errors="coerce")

rows = []
for dt, g in df2.groupby("datetime", sort=True):
    if pd.isna(dt):
        continue
    idx  = g["confidence_mass_sum"].astype(float).idxmax()
    best = df2.loc[idx]
    rows.append({
        "capture_datetime": dt,
        "capture_date": best.get("capture_date"),
        "capture_time": best.get("capture_time"),
        "confidence_mass_max": float(best["confidence_mass_sum"]),
        "pred_count_max": int(best["total_pred_count"]),
        "detection_rate_max": float(best["detection_rate"]),
        "focus_selected": best["focus"],
        "num_focus_available": int(g["focus"].nunique(dropna=True)),
        "num_focus_detected": int((g["detection_rate"] > 0).sum()),
        "any_focus_detected": int((g["detection_rate"] > 0).any()),
        "confidence_mass_range_across_focus": float(g["confidence_mass_sum"].max() - g["confidence_mass_sum"].min()),
        "confidence_mass_std_across_focus": float(g["confidence_mass_sum"].std(ddof=0)) if len(g) > 1 else 0.0,
    })

out = pd.DataFrame(rows).sort_values("capture_datetime")
try:
    out.to_csv(OUT_TABLE3_FUSE, index=False)
    logger.info(f"Step 3 完成：table3={out.shape}，保存 {OUT_TABLE3_FUSE}")
except Exception as e:
    logger.error(f"保存 table3 失败: {e}")

# ── 5. 验证 ────────────────────────────────────────────────

In [ ]:
# ── 验证：抽查三张表 head ─────────────────────────────────────
from IPython.display import display

logger.info(f"验证：table1={img_df.shape}, table2={agg.shape}, table3={out.shape}")
print("\n── table1 per-image ──")
display(img_df.head(3))
print("\n── table2 per-timepoint × focus ──")
display(agg.head(3))
print("\n── table3 per-timepoint fused ──")
display(out.head(3))

# ── [可选] 快速可视化 ────────────────────────────────────────

In [ ]:
# 可选：快速绘制预测数量时序图
import plotly.express as px

fig = px.line(agg, x='datetime', y='total_pred_count',
              color='focus', title='Total Prediction Count Over Time (by focus)')
fig.show()